In [ ]:
from __future__ import annotations

import re
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Iterable, Optional, Literal

TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

DEFAULT_HEADING_PATTERNS = [
    r"related\s+work(s)?",
    r"related\s+research",
    r"prior\s+work",
    r"previous\s+work",
    r"literature\s+review",
    r"background",
    r"state\s+of\s+the\s+art",
]

# Whole subtrees to exclude if present as blocks (less common for GROBID body, but safe)
EXCLUDE_SUBTREES = {"figure", "table", "formula", "graphic"}

FigurePolicy = Literal["strip_refs", "drop_paragraphs", "keep"]


def _local_name(tag: str) -> str:
    return tag.split("}", 1)[-1] if "}" in tag else tag


def _prune_excluded_subtrees(root: ET.Element) -> None:
    for parent in root.iter():
        for child in list(parent):
            if _local_name(child.tag) in EXCLUDE_SUBTREES:
                parent.remove(child)


def _has_ref_type(el: ET.Element, ref_type: str) -> bool:
    # Matches <ref type="figure"...> or <ref type="table"...>
    for r in el.findall(".//tei:ref", TEI_NS):
        if (r.get("type") or "").lower() == ref_type.lower():
            return True
    return False


def _remove_refs_by_type_inplace(el: ET.Element, ref_types: set[str]) -> None:
    """
    Remove <ref> nodes with @type in ref_types, preserving their tail text.
    """
    ref_types_lc = {t.lower() for t in ref_types}
    for parent in el.iter():
        for child in list(parent):
            if _local_name(child.tag) == "ref" and (child.get("type") or "").lower() in ref_types_lc:
                # preserve tail text so words after the ref aren't lost
                tail = child.tail or ""
                idx = list(parent).index(child)
                parent.remove(child)
                # attach tail either to previous sibling or parent.text
                if idx > 0:
                    prev = list(parent)[idx - 1]
                    prev.tail = (prev.tail or "") + tail
                else:
                    parent.text = (parent.text or "") + tail


def _string_value(el: ET.Element) -> str:
    # NO normalization / cleaning
    return "".join(el.itertext())


def extract_related_work_from_tei(
    tei_xml: str,
    heading_patterns: Optional[Iterable[str]] = None,
    include_lists: bool = True,
    figure_policy: FigurePolicy = "strip_refs",
    return_blocks: bool = False,
) -> str | list[str] | None:
    """
    Extract related-work-like section from TEI, while preventing figure/table
    references like: <p>Fig. <ref type="figure" ...>4</ref> ...</p>

    figure_policy:
      - "strip_refs": remove <ref type="figure/table"> nodes, keep paragraph text
      - "drop_paragraphs": drop any <p> that contains figure/table refs
      - "keep": leave refs in (not recommended for your goal)
    """
    patterns = list(heading_patterns or DEFAULT_HEADING_PATTERNS)
    head_any_re = re.compile("|".join(patterns), flags=re.IGNORECASE)

    root = ET.fromstring(tei_xml)
    body = root.find(".//tei:text/tei:body", TEI_NS)
    if body is None:
        return None

    candidate_div = None
    for div in body.findall(".//tei:div", TEI_NS):
        head = div.find("tei:head", TEI_NS)
        if head is None:
            continue

        head_text = _string_value(head)

        # Use a simplified version ONLY for matching (does not affect returned text)
        simplified = head_text.lower()
        simplified = re.sub(r"^[\d.\s]+", "", simplified)
        simplified = re.sub(r"[:\-].*$", "", simplified).strip()

        if head_any_re.search(simplified) or head_any_re.search(head_text):
            candidate_div = div
            break

    if candidate_div is None:
        return None

    # Remove block-level figures/tables if present
    _prune_excluded_subtrees(candidate_div)

    blocks: list[str] = []

    # Extract paragraphs in order
    for p in candidate_div.findall(".//tei:p", TEI_NS):
        has_fig = _has_ref_type(p, "figure")
        has_tbl = _has_ref_type(p, "table")

        if figure_policy == "drop_paragraphs" and (has_fig or has_tbl):
            continue

        if figure_policy == "strip_refs":
            # Work on a shallow copy of the <p> element to avoid mutating original tree unexpectedly
            p_copy = ET.fromstring(ET.tostring(p, encoding="utf-8"))
            _remove_refs_by_type_inplace(p_copy, {"figure", "table"})
            txt = _string_value(p_copy)
        else:
            txt = _string_value(p)

        # Keep text as-is (no strip/cleanup)
        if txt:
            blocks.append(txt)

    if include_lists:
        for item in candidate_div.findall(".//tei:list/tei:item", TEI_NS):
            if figure_policy == "strip_refs":
                item_copy = ET.fromstring(ET.tostring(item, encoding="utf-8"))
                _remove_refs_by_type_inplace(item_copy, {"figure", "table"})
                txt = _string_value(item_copy)
            else:
                txt = _string_value(item)

            if figure_policy == "drop_paragraphs" and (
                _has_ref_type(item, "figure") or _has_ref_type(item, "table")
            ):
                continue

            if txt:
                blocks.append(txt)

    if not blocks:
        txt = _string_value(candidate_div)
        return [txt] if (txt and return_blocks) else (txt or None)

    return blocks if return_blocks else "\n\n".join(blocks)


def extract_related_work_from_file(path: str | Path, **kwargs):
    xml = Path(path).read_text(encoding="utf-8", errors="replace")
    return extract_related_work_from_tei(xml, **kwargs)

In [2]:
import os
import json 
from tqdm import tqdm

nlpeer_path = "ARR-22/data"
total_files = []

for folders in tqdm(os.listdir(nlpeer_path)):
    if folders != 'meta.json':
        for filename in os.listdir(os.path.join(nlpeer_path, folders, "v1")):
            if filename.endswith(".tei"):
                tei_path = os.path.join(nlpeer_path, folders, "v1", filename)
                related_work = extract_related_work_from_file(tei_path, figure_policy="strip_refs")
                if related_work != None:
                    total_files.append(related_work)

with open("nlpeer_related_work.json", "w", encoding="utf-8") as f:
    json.dump(total_files, f, indent=2)


100%|██████████| 477/477 [00:18<00:00, 26.50it/s]
